In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
text1 = 'sentence-transformers embedding model'
text2 = 'sentence-transformers'

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
embedding1 = model.encode(text1) #stored in vectordb
embedding2 = model.encode(text2)
#print(len(embedding))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
text1 = 'I love Artificial Intelligence'
text2 = 'I like Machine Learning'
emb1 = model.encode(text1)
emb2 = model.encode(text2)
print(cosine_similarity([emb1],[emb2]))

[[0.71532476]]


In [ ]:
pip install faiss-cpu

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning is a type of machine learning based on neural networks.",
    "Python is widely used for data science and machine learning.",
    "Hadoop is used for distributed storage and big data processing.",
    "Spark is faster than Hadoop for in-memory data processing."
]

In [ ]:
embedding=embedding_model.encode(documents)
embedding

array([[-0.02345927, -0.01058112,  0.07192818, ...,  0.06038335,
         0.07951809, -0.0471024 ],
       [-0.06388477, -0.04189363,  0.06170243, ...,  0.06798722,
         0.05373019,  0.01341107],
       [-0.07433756, -0.02514078, -0.01920343, ...,  0.08415696,
         0.12070204,  0.04112899],
       [-0.0596775 ,  0.05047086, -0.03874555, ..., -0.00949509,
         0.00612472,  0.06994062],
       [-0.05471694,  0.0020039 , -0.0241049 , ..., -0.03662821,
         0.01922276,  0.02972482]], dtype=float32)

In [ ]:
dim=embedding.shape[1]
dim

384

In [ ]:
index = faiss.IndexFlatL2(dim)  #L2-similarity search,fix the dimension
index.add(np.array(embedding))  #all the vector embeddings stored in index like object
print(index)

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x77fedd5c91a0> >


In [ ]:
generator = pipeline('text-generation',model='gpt2') #pipeline ->it is used for user query, text-generation->transformer model in that we used gpt2 model

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ##step:5 RAG Function
# def rag_query(query,top_k=2):
#   #convert query to embedding
#   query_embedding=embedding_model.encode([query])
#   #search similar docs
#   distances,indices = index.search(np.array(query_embedding),top_k)
#   print(distances,indices)
#   print(indices[0])
#   #Retrieve relevant  documnets
#   retrieved_docs = [documents[i] for i in indices[0]]
#   #create agumented prompt
#   context = " ".join(retrieved_docs)
#   prompt = f"Answer the question based on the context.\n\n\nContext:{context}.\n\n\nQuery:{query}.\n\n\nAnswer:"
#   #Generator answer
#   result = generator(prompt,max_length=150,num_return_sequences=1)
#   return result[0]['generated_text']


# response=rag_query('what is deep learning?')
# print(response)

In [ ]:
# ----------------------------
# STEP 5: RAG Function
# ----------------------------
def rag_query(query, top_k=2):

    # Convert query to embedding
    query_embedding = embedding_model.encode([query])

    # Search similar docs
    distances, indices = index.search(np.array(query_embedding), top_k)
    print(distances,indices)
    print(indices[0])
    # Retrieve relevant documents
    retrieved_docs = [documents[i] for i in indices[0]]

    # # Create augmented prompt
    context = " ".join(retrieved_docs)
    prompt = f"Answer the question based on the context.\n\nContext: {context}\n\nQuestion: {query}\nAnswer:"

    # # Generate answer
    result = generator(prompt, max_length=150, num_return_sequences=1)

    return result[0]["generated_text"]


# ----------------------------
# STEP 6: Test Query
# ----------------------------
response = rag_query("What is deep learning?")
print(response)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[[0.34885886 0.91943586]] [[1 0]]
[1 0]
Answer the question based on the context.

Context: Deep learning is a type of machine learning based on neural networks. Machine learning is a subset of artificial intelligence.

Question: What is deep learning?
Answer: Deep learning is a type of machine learning based on neural networks. Machine learning is a subset of artificial intelligence.

Question: How do you measure the results of deep learning?

Answer: You measure the results of deep learning using the algorithm that you use to train the machine.

The algorithm is a recurrent neural network (RNN), which is a neural network that can be trained by a few different types of neural networks.

Now, here's an example which describes how machine learning looks like. Let's say that you want to measure the distance from a point to a point. A data point is an object in a set of data. Every time you go to a data point, there is a new point in the set of data, and a new point in the set of data. In

#POS tagging in rag

In [ ]:
# ----------------------------
# Imports
# ----------------------------
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

# ----------------------------
# NLTK Downloads
# ----------------------------
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')

# ----------------------------
# Lemmatizer
# ----------------------------
lemmatizer = WordNetLemmatizer()

# ----------------------------
# POS Mapping Function
# ----------------------------
def get_pos(tag):
    if tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

# ----------------------------
# Query Preprocessing Function
# ----------------------------
def preprocess_query(query):
    tokens = word_tokenize(query)
    tagged = pos_tag(tokens)

    lemmatized = [
        lemmatizer.lemmatize(word, get_pos(pos))
        for word, pos in tagged
    ]

    return " ".join(lemmatized)

# ----------------------------
# STEP 5: RAG Function
# ----------------------------
def rag_query(query, top_k=2):

    # Convert query to embedding
    query_embedding = embedding_model.encode([query])

    # Search similar docs
    distances, indices = index.search(np.array(query_embedding), top_k)
    print(distances, indices)
    print(indices[0])

    # Retrieve relevant documents
    retrieved_docs = [documents[i] for i in indices[0]]

    # Create augmented prompt
    context = " ".join(retrieved_docs)
    prompt = f"Answer the question based on the context.\n\nContext: {context}\n\nQuestion: {query}\nAnswer:"

    # Generate answer
    result = generator(prompt, max_length=150, num_return_sequences=1)

    return result[0]["generated_text"]

# ----------------------------
# STEP 6: Test Query
# ----------------------------
user_query = "What is deep learning?"
processed_query = preprocess_query(user_query)

response = rag_query(processed_query)
print(response)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/

[[0.38657054 0.93437254]] [[1 0]]
[1 0]
Answer the question based on the context.

Context: Deep learning is a type of machine learning based on neural networks. Machine learning is a subset of artificial intelligence.

Question: What be deep learning ?
Answer: Deep learning is a set of algorithms that are used to solve complex problems. The algorithms are usually made up of a set of algorithms, which are called deep learning components. Deep learning is used by machine learning to learn new ideas.

You should also know that deep learning is a bit different than deep learning in that it is a type of neural network (a type of machine learning). Deep learning is mostly based on the idea that a machine learning algorithm can learn an idea, or learn something new.

Examples of deep learning components:

There are also artificial intelligence components. These neural networks are based on a set of artificial intelligence (AI) algorithms. These artificial intelligence components are built on